Copyright (c) Qualcomm Technologies, Inc. and/or its subsidiaries.<br>
SPDX-License-Identifier: BSD-3-Clause-Clear

**Takeaways:** Users will learn how to tune the Resnet50 model for best throughput and latency

**Before you start:** 
- There are some commands (folder locations etc) that will need to be updated in this notebook based on the platform and installation location. Some commands might need sudo prefix to run properly.
- The terms 'model' and 'network' are used interchangeably in this notebook. 
- The terms 'NSP' and 'AI compute core" are used interchangeably in this notebook.

**Last Verified Qualcomm Cloud AI Platform SDK and Apps SDK Version:** Platform SDK 1.23.2.0 and Apps SDK 1.23.2.0

# <span style='color:Blue'> Performance Tuning on Cloud AI </span>

##  Pre-requisite reading 
New users on Cloud AI platforms are expected to go over the Cloud AI SoC architecture and the key compile/runtime parameters that determine performance. This is discussed in the Tune Performance section in the Inference workflow documentation. 


## Introduction 
This notebook is for beginners and will take the user through the workflow to achieve best throughput and latency on Cloud AI platforms for the Resnet50 model. 

Here is the workflow that will be demonstated in this notebook. 

1. **Install required packages**: Begin by installing all the required packages
2. **Import the model**: Download the Resnet50 model in ONNX. 
2. **Device Health Check**: Query the device health using qaic-util tool. 
2. **Identify best throughput configuration**: Go over the Model Configurator tool to identify best throughput 
3. **Identity least latency configuration**: Go over the key parameters to tweak for least latency


# <span style='color:Blue'> 1. Install required packages </span>

We will install the required Python packages 

In [ ]:
%pip install -r requirements.txt

# <span style='color:Blue'>2. Download the model </span>

Download the pretrained Resnet50 model. 

In [ ]:
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!{sys.executable} -m optimum.commands.optimum_cli export onnx --model 'microsoft/resnet-50' resnet-50_onnx/ --opset 11 --task image-classification --width 224 --height 224 --num_channels 3

# <span style='color:Blue'> 3. Check device health </span>

'qaic-util' tool can be used to query the health of all the Cloud AI cards in the server. (Ensure that the command is run with sudo or this jupyter notebook is run with --allow-root and sudo)

"Status: Ready" indicates that the health of the cards is good. 
"Status: Error" indicates that the cards are not in good health and a system administrator needs to be contacted to rectify the issue. 

In [ ]:
!/opt/qti-aic/tools/qaic-util -q | grep "Status"

# <span style='color:Blue'> 4. Identify the best throughput configuration </span>

Model Configurator is a python script that is used to find the optimal configuration of batch size, cores etc for a given model for the throughput (Inf/s). The input to model configurator is the model and a search space that the tool iterates over.   

1. Use the optimal config (model configurator output) as an indicator of the best compile flags.
2. Compile with qaic-compile and run with qaic-runner using optimal config from step 1 to validate the performance. 

Additional details on the Parameters- https://qualcomm.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Inference-Workflow/model-compilation/Tune-performance/

Additional Documentation on Model Configurator tool here- https://docs.qualcomm.com/bundle/publicresource/topics/80-PT790-993B/network-perform-tuning.html#network-perform-tuning_qaic-model-configurator

Key compile optimization flags are 
- **aic-num-cores** : # of AI compute cores (aka NSP) used to compile the model
- **bs** : batch size
- **mos** : Maximum output splitting, denotes the no of AI compute cores across which an output channel (and associated weights) is split. 
- **ols** : Overlap splitting, enabled output splitting to improve core level parallelism, eg tensor and vector unit

Key runtime optimization flags are 
- **instance/activations** : No of instances of the compiled binary that can be run based on the # of AI compute cores on the card 
- **set-size** : denotes the number of inferences per instance that can be queued up on the host. Hides host side overhead by pipelining inferences. CV models typically can benefit from higher set-sizes for higher throughput (with increased latency).  

## Config Optimizer 
The optimized search is run on one or more searchable parameters (aic-num-cores, instances, batch-size etc). The search space is provided through a json configuration file. 
This table captures the key elements of the configuration file. 

| Key | Type | Description | Recommended value|
| :- |:- | :- | :-|
|"max_func_eval"|Integer|Maximun number of evaluations to do for each initial point. This number can be increased if successful convergence is not achieved|200|
|"Objective"|String| Search objective. Options are "max-throughput" or "min-latency" | Choose "max-throughput" for maximum throughput or "min-latency" for minimum latency |
|“params”|Json Object|Provide the search range for each of the parameters - cores, mos, ols, etc. through min, max values| See table below |
“initial_values”|List of Json Objects|List of initial values for the search parameters. A fresh search is initiated from each of these points and the results returned. Initial Values must be picked from within the search range defined in “params”|	Provide multiple initial values as shown in the example json|
|“static_params”|Json Object|Optional static values to be used for searchable parameters which have been excluded from the search space| |


### Parameter Range


| Parameter|	Recommended Search Range|	Valid Range |	Comments|
| :- | :- | :- | :-|
|cores|	|	1-14(Standard Card), 1-16 (Pro/Ultra Card)| cores*instances should be less than or equal to the number of NSP's on the device 1-14(Standard Card), 1-16 (Pro/Ultra Card)|	
|instances|	|1-14(Standard Card), 1-16 (Pro/Ultra Card)| cores*instances should be less than or equal to the number of NSP's on the device 1-14(Standard Card), 1-16 (Pro/Ultra Card)
|mos	|1-Number of NSP on device|	Number of NSP's on device||	
|ols	|1-8|	Integers>0|	
|batch-size (bs)|	1-16|	Integers>0|	Min, max values must be power of 2. The max value would depend on the model|
	

In [ ]:
#Lets assume the max Number of NSP on the device is 16. 
!cat resnet_base_dopt_throughput.json

In [ ]:
# Dump the model inputs and outputs 

import onnx
model = onnx.load("resnet-50_onnx/model.onnx")
for _input in model.graph.input:
    print(_input)
for _output in model.graph.output:
    print(_output)

In [ ]:
# Run model_configurator tool to identify the best throughput configuration. 
# For CV networks, higher set sizes are preferred for higher throughput. Default set-size is 10. 
!{sys.executable} /opt/qti-aic/scripts/qaic-model-configurator/model_configurator.py -model resnet-50_onnx/model.onnx \
-onnx-define-symbol-batch-size=batch_size \
-onnx-define-symbol=num_channels,3 -onnx-define-symbol=height,224  -onnx-define-symbol=width,224 \
-multicast-weights \
-config=resnet_base_dopt_throughput.json -max-parallel-compilations=16 -time=5 \
-convert-to-fp16 -device-id=0 

## Measure end-to-end latency 

Here are the steps to measure the end-to-end latency as well as the breakdown of latency for the best configuration identified in the previous step. (Configurations depend on the card variant that the test is being run on). 

1. Compile the model using 'qaic-compile' with the configuration parameters identified in previous step using model configurator. 
2. Execute the compiled model using 'qaic-runner' with the run time parameters identified in previous step using model configurator. Run 'qaic-runner' with flags that dump latency information.
3. Post process the latency information to identify percentile distribution (mean, median, 95 and 99) in latency across inferences.

In [ ]:
## Compile the model 

!rm -rf compiled_fp16
!/opt/qti-aic/exec/qaic-compile -v -aic-hw  \
-m=resnet-50_onnx/model.onnx \
-onnx-define-symbol=batch_size,4 \
-onnx-define-symbol=num_channels,3 -onnx-define-symbol=height,224  -onnx-define-symbol=width,224 \
-mos=1 -ols=2 -aic-num-cores=2 \
-stats-batchsize=4 -aic-binary-dir=./compiled_fp16 \
-multicast-weights -convert-to-fp16 \
-aic-hw-version=ai100

In [ ]:
## Execute the compiled model with the latency flags. qaic-runner generates random data if input data is not passed to it. 
!mkdir resnet50_stats
!/opt/qti-aic/exec/qaic-runner --test-data ./compiled_fp16  -d 0 -a 7 -S 10 \
--aic-profiling-type latency --aic-profiling-out-dir ./resnet50_stats \
--aic-profiling-start-iter 100 --aic-profiling-num-samples 9999 --time 20 

### Latency breakdown 

The end-to-end inference latency can be broken down into 4 major categories - Application, Linux Runtime (LRT) processing, Kernel mode driver (KMD) processing and Cloud AI device processing. 


![Latency Breakdown](Images/Latency.jpg)

**Key Latency Stats** (units in us)
- *totalRoundtripTime* :Time from point where application (qaic-runner in this case) calls Runtime API and ends where post-processing is complete and control is returned to the application indicating the inference is complete
- *preProcTime* : Time taken to pre-process the data (model input) on the host
- *postProc* : Time taken to post-process the data (model output) on the host
- *execTotal* : Time from inference object being submitted to kernel, completion on hardware and processing is returned to user-space


In [ ]:
## Post process the latency information to identify percentile distribution

!{sys.executable} latency_stats_python3.py ./resnet50_stats/aic-profiling-program-0-latency.txt \
./resnet50_stats/aic-profiling-program-1-latency.txt \
./resnet50_stats/aic-profiling-program-2-latency.txt \
./resnet50_stats/aic-profiling-program-3-latency.txt \
./resnet50_stats/aic-profiling-program-4-latency.txt \
./resnet50_stats/aic-profiling-program-5-latency.txt \
./resnet50_stats/aic-profiling-program-6-latency.txt 

As observed, for the configuration choosen, throughput of ~4380 images/sec is observed with a mean latency of ~64ms per batched (bs=4) inference. 

# <span style='color:Blue'> 5. Identify the least latency configuration </span>

Identifying the least latency configurations requires the users to run the model_configurator tool with the "objective" parameter set to "minimize_latency". 

Minimum latency is achieved when the batch-size, instances and set-size are set to 1. 
We need to iterate through the cores used to compile the network to identify the least latency config. 

The key difference in the dopt.json files used for best throughput vs least latency are in the initial values. The number of instances is always set to 1 for least latency. For best throughput, the initial value of instances = floor((total no of cores in the device) / (no of cores used to compile the model)) 

Lets go through these steps for the resnet50 model.

In [ ]:
!{sys.executable} /opt/qti-aic/scripts/qaic-model-configurator/model_configurator.py -model resnet-50_onnx/model.onnx  \
-onnx-define-symbol-batch-size=batch_size \
-onnx-define-symbol=num_channels,3 -onnx-define-symbol=height,224  -onnx-define-symbol=width,224 \
-multicast-weights \
-config=resnet_base_dopt_min_latency.json -max-parallel-compilations=16 -time=5 \
-convert-to-fp16 -device-id=0 -search-set-size 1

The Configuration with 9 cores and batch size 1 is returned as the configuration that provides least latency per image. Developers could also use row 3 which uses 7 cores. On a device with 16 cores, users can run 2 instances with a slight increase in latency.(Configurations depend on the card variant that the test is being run on)

In [ ]:
## aic-num-cores =7 and instances = 1
!rm -rf compiled_fp16
!rm -rf resnet50_stats
!mkdir resnet50_stats

!/opt/qti-aic/exec/qaic-compile -v -aic-hw  \
-m=resnet-50_onnx/model.onnx \
-onnx-define-symbol=batch_size,1 \
-onnx-define-symbol=num_channels,3 -onnx-define-symbol=height,224  -onnx-define-symbol=width,224 \
-mos=1 -ols=1 -aic-num-cores=7 \
-stats-batchsize=1 -aic-binary-dir=./compiled_fp16 \
-multicast-weights -convert-to-fp16 \
-aic-hw-version=ai100

In [ ]:
# Running a single instance of the compiled binary
!/opt/qti-aic/exec/qaic-runner --test-data ./compiled_fp16  -d 0 -a 1 -S 1 \
--aic-profiling-type latency --aic-profiling-out-dir ./resnet50_stats \
--aic-profiling-start-iter 100 --aic-profiling-num-samples 99999 --time 10 

In [ ]:
# Measure latency for the single instance
!{sys.executable} latency_stats_python3.py ./resnet50_stats/aic-profiling-program-0-latency.txt 

In [ ]:
# Running 2 instances of the compiled binary
!/opt/qti-aic/exec/qaic-runner --test-data ./compiled_fp16  -d 0 -a 2 -S 1 \
--aic-profiling-type latency --aic-profiling-out-dir ./resnet50_stats \
--aic-profiling-start-iter 100 --aic-profiling-num-samples 99999 --time 10 

In [ ]:
# Measure latency for 2 instances
!{sys.executable} latency_stats_python3.py ./resnet50_stats/aic-profiling-program-0-latency.txt \
./resnet50_stats/aic-profiling-program-1-latency.txt 

We see that the throughput has increased significantly (~680 -> ~1360 inf/s) while latency has also increased but only slightly (~~1.428ms to ~1.435ms). 